# Setup and the AgentChat API

**01. Introduction to AutoGen** positioned AgentChat as the high-level API for multi-agent apps. This notebook goes hands-on: **install** the 0.4 packages, understand the **core / agentchat / ext** split, configure **`OpenAIChatCompletionClient`**, manage the **model client lifecycle**, run **`asyncio`** correctly in Jupyter, set **environment variables**, execute your first **`AssistantAgent.run()`**, stream with **`Console`**, and run an **offline fake client** when no API key is available.

Prerequisites: **01. Introduction to AutoGen**. Next: **03. AssistantAgent and ConversableAgent**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import importlib.metadata as im
import os

os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

for pkg in ("autogen-agentchat", "autogen-ext", "autogen-core"):
    try:
        print(f"{pkg}:", im.version(pkg))
    except im.PackageNotFoundError:
        print(f"{pkg}: not installed — pip install autogen-agentchat autogen-ext[openai]")

---

## 1. Installation — AgentChat 0.4 Packages

Microsoft split AutoGen into focused PyPI packages:

```bash
pip install "autogen-agentchat" "autogen-ext[openai]"
```

| Package | Install name | Purpose |
|---------|--------------|--------|
| Core runtime | `autogen-core` | Pulled transitively; actors, cancellation |
| AgentChat API | `autogen-agentchat` | Agents, teams, termination, UI helpers |
| Extensions | `autogen-ext[openai]` | OpenAI model client, executors, integrations |

**Do not** `pip install pyautogen` for this track — post-0.2.34 PyPI releases may not be from Microsoft. Pin `autogen-agentchat>=0.4` in `requirements.txt`.

---

## 2. Package Split — Core vs AgentChat vs Ext

```
┌─────────────────────────────────────────────────────────┐
│                    Your application                      │
└───────────────────────────┬─────────────────────────────┘
                            │
              ┌─────────────▼─────────────┐
              │     autogen-agentchat      │  ◄── this track
              │  AssistantAgent, teams, UI │
              └─────────────┬─────────────┘
                            │
              ┌─────────────▼─────────────┐
              │       autogen-ext          │
              │ OpenAIChatCompletionClient │
              └─────────────┬─────────────┘
                            │
              ┌─────────────▼─────────────┐
              │       autogen-core         │
              │  CancellationToken, events │
              └───────────────────────────┘
```

Most tutorials need only **agentchat** + **ext**. Build custom frameworks on **core** only if AgentChat is too opinionated.

In [ ]:
import autogen_agentchat
import autogen_core

try:
    import autogen_ext
    print("autogen_ext: OK")
except ImportError:
    print("autogen_ext: missing — pip install autogen-ext[openai]")

print("agentchat module:", autogen_agentchat.__name__)
print("core module:", autogen_core.__name__)

---

## 3. Environment Variables and API Keys

| Variable | Role |
|----------|------|
| `OPENAI_API_KEY` | Default key for `OpenAIChatCompletionClient` |
| `OPENAI_BASE_URL` | Optional compatible API base (local LLM gateways) |

In notebooks, set the key in the first code cell (placeholder above) **or** load from `.env` with `python-dotenv` in your app — never commit real keys.

In [ ]:
key_from_env = os.environ.get("OPENAI_API_KEY", "")
print("key configured:", bool(key_from_env) and not key_from_env.startswith("sk-proj-placeholder"))

---

## 4. OpenAIChatCompletionClient

Replaces v0.2 `llm_config`. The client is **async** and **reusable** across multiple agents:

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
)
print("client type:", type(model_client).__name__)

---

## 5. Model Client Lifecycle

| Phase | Action |
|-------|--------|
| **Create** | One client per process (or per request in serverless — see **16**) |
| **Share** | Pass the same `model_client` to multiple `AssistantAgent` instances |
| **Close** | `await model_client.close()` when the session ends |

Failing to close clients can leak HTTP connections in long-running FastAPI workers.

In [ ]:
from autogen_agentchat.agents import AssistantAgent


async def client_lifecycle_demo() -> None:
    client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    try:
        agent_a = AssistantAgent("a", model_client=client)
        agent_b = AssistantAgent("b", model_client=client)
        print("two agents created with shared client")
    finally:
        await client.close()
        print("client closed")


await client_lifecycle_demo()

---

## 6. Asyncio Event Loop in Notebooks

AgentChat methods are **coroutines**. Patterns:

| Context | Pattern |
|---------|--------|
| **Jupyter** | Top-level `await coroutine()` (IPython async REPL) |
| **Python script** | `asyncio.run(main())` |
| **FastAPI** | `await agent.run(...)` inside `async def` routes |

Avoid `asyncio.get_event_loop().run_until_complete()` in modern Python 3.10+ scripts — use `asyncio.run()`.

In [ ]:
async def notebook_pattern() -> str:
    return "async works in Jupyter"


print(await notebook_pattern())

---

## 7. Notes API Corpus

Shared teaching data for agents and tools in this track:

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("Notes corpus:", list(NOTES.keys()), "| chunks:", [c["id"] for c in DOC_CHUNKS])

---

## 8. First AssistantAgent.run()

The simplest end-to-end AgentChat call:

In [ ]:
async def first_run() -> None:
    client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    agent = AssistantAgent(
        name="notes_assistant",
        model_client=client,
        system_message="Answer in one short sentence using the Notes API context when relevant.",
    )
    result = await agent.run(task="Where do Notes API routes live?")
    print(result.messages[-1].content)
    await client.close()


await first_run()

### 8.1 TaskResult Shape

`run()` returns **`TaskResult`** with:

- **`messages`** — full conversation transcript for this run
- **`stop_reason`** — why the agent stopped (completed, termination, error)

Equivalent lower-level API: `on_messages()` with explicit `TextMessage` list (**03**).

---

## 9. Streaming with run_stream and Console

For live output, stream messages and pipe to **`Console`**:

In [ ]:
from autogen_agentchat.ui import Console


async def stream_demo() -> None:
    client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    agent = AssistantAgent("streamer", model_client=client)
    stream = agent.run_stream(task="Say hello to the Notes API team.")
    await Console(stream)
    await client.close()


await stream_demo()

`Console` renders agent messages as they arrive — ideal for demos and debugging. Production apps consume `run_stream` directly and forward tokens over WebSockets.

---

## 10. on_messages — Explicit Message List

`run(task=...)` is sugar over **`on_messages()`**:

In [ ]:
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken


async def on_messages_demo() -> None:
    client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    agent = AssistantAgent("explicit", model_client=client)
    response = await agent.on_messages(
        [TextMessage(content="What header carries JWT tokens?", source="user")],
        cancellation_token=CancellationToken(),
    )
    print(response.chat_message.content)
    await client.close()


await on_messages_demo()

---

## 11. Offline Demo — Fake Model Client

Learn the agent wiring without spending tokens. Mock at the client boundary:

In [ ]:
class FakeChatCompletionClient:
    """Minimal stub for offline teaching demos."""

    def __init__(self, canned: str) -> None:
        self._canned = canned

    async def create(self, messages, **kwargs):
        class _Msg:
            content = self._canned

        class _Choice:
            message = _Msg()

        class _Response:
            choices = [_Choice()]

        return _Response()

    async def close(self) -> None:
        pass


async def offline_agent_demo() -> None:
    fake_client = FakeChatCompletionClient(canned=NOTES["n1"])
    print("offline canned response:", fake_client._canned)


await offline_agent_demo()

For CI pipelines, mock `OpenAIChatCompletionClient` with `unittest.mock.AsyncMock` or use recorded responses — details in **15. Observability and Debugging**.

---

## 12. CancellationToken

Long runs should respect cancellation (user clicked Stop, request timeout):

In [ ]:
async def cancellable_run() -> None:
    client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    agent = AssistantAgent("worker", model_client=client)
    token = CancellationToken()
    response = await agent.on_messages(
        [TextMessage(content="Quick: what is pytest conftest.py for?", source="user")],
        cancellation_token=token,
    )
    print(response.chat_message.content[:80], "...")
    await client.close()


await cancellable_run()

---

## 13. AgentChat vs Legacy pyautogen 0.2

| Task | v0.2 | AgentChat 0.4 |
|------|------|---------------|
| Import agents | `from autogen import AssistantAgent` | `from autogen_agentchat.agents import AssistantAgent` |
| Configure LLM | `llm_config={"config_list": [...]}` | `OpenAIChatCompletionClient(...)` |
| Start chat | `user.initiate_chat(assistant, message=...)` | `await assistant.run(task=...)` |
| Group chat | `GroupChat` + `GroupChatManager` | `RoundRobinGroupChat([...])` |

Full mapping table: **03. AssistantAgent and ConversableAgent**.

---

## 14. Cross-Reference — LangGraph Setup

LangGraph notebooks use `ChatOpenAI` + `StateGraph` (**02. LangGraph/02**). AutoGen uses **`model_client`** + **`AssistantAgent`**. Both need `OPENAI_API_KEY`. LangGraph adds **checkpointers**; AutoGen adds **teams** — pick the orchestration layer that matches your product.

---

## 15. Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `PackageNotFoundError` | Wrong package name | `pip install autogen-agentchat autogen-ext[openai]` |
| `RuntimeWarning: coroutine was never awaited` | Missing `await` | Prefix with `await` or use `asyncio.run()` |
| 401 from OpenAI | Bad or placeholder key | Set real `OPENAI_API_KEY` |
| Import from `autogen` fails | v0.2 tutorial | Switch imports to `autogen_agentchat` |
| Hang on `Console` | Waiting for team turn | Check termination condition (**13**) |

---

## 16. Key Imports Reference

```python
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
```

---

## 17. Learning Path — This Series

```
01 Introduction ──► 02 Setup/AgentChat API ──► 03 AssistantAgent
     │
     ▼
04 UserProxy/HITL ──► 05 Two-agent patterns ──► 06 Tools
     │
     ▼
07 Code execution ──► 08 GroupChat teams ──► 09 Speaker selection
     │
     ▼
10 Custom roles ──► 11 Sequential/hierarchical ──► 12 Memory/state
     │
     ▼
13 Termination/guardrails ──► 14 AutoGen Studio ──► 15 Observability
     │
     ▼
16 Production patterns
```

Each notebook is **self-contained** but follows this sequence. Framework comparisons live in **`00. Frameworks/07. AutoGen`**. LangGraph checkpointing and graph interrupts are covered in **`02. LangGraph/09`**.

---

## 18. Common Beginner Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Using legacy `pyautogen` 0.2 imports | Wrong API, broken tutorials | Install `autogen-agentchat` 0.4+ |
| Forgetting `await` on `run()` | Coroutine never executes | Use `async def` + `await` or `asyncio.run()` |
| Sharing one `model_client` incorrectly | Leaked connections in prod | `await model_client.close()` when done |
| No termination condition on teams | Unbounded LLM spend | `TextMentionTermination` / `MaxMessageTermination` (**13**) |
| Blocking `input()` in async servers | Event loop stalls | Async `input_func` or queue-based HITL (**04**) |
| AutoGen for rigid state machines | Hard-to-audit flows | Use **LangGraph** when graph = contract |

---

## 19. Summary

AgentChat **0.4** setup centers on three packages (**core**, **agentchat**, **ext**), an **`OpenAIChatCompletionClient`** shared across agents, and **async** `run()` / `run_stream()` / `Console` for execution.

Key takeaways:

- Install `autogen-agentchat` + `autogen-ext[openai]`, not legacy `pyautogen`.
- Create one **model client**, pass to agents, **`await close()`** when done.
- Use top-level **`await`** in Jupyter; **`asyncio.run()`** in scripts.
- **`run(task=...)`** is the primary entry point; **`on_messages`** for explicit control.
- **`Console(stream)`** streams team/agent output interactively.
- Mock the client boundary for **offline** tests and demos.

Next: **03. AssistantAgent and ConversableAgent** — parameters, tools, `reflect_on_tool_use`, and the v0.2 migration table.